# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OsaidKamran/FLYRANK.AI-SUMMER-INTERNSHIP-MACHINE-LEARNING_UPDATED/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This capstone lane is unsupervised: there is no ground-truth label for "content archetype," so the task is dimensionality reduction followed by clustering rather than classification. I am using an Autoencoder to compress the pooled numerical SEO signals (impressions, clicks, CTR, average position, and engagement-rate features) into a dense latent space, then running K-Means on that latent space to discover the archetypes. I chose an Autoencoder over flat PCA because these signals are noisy and likely interact non-linearly — a page that is old-but-authoritative behaves differently from a page that is new-but-viral, and PCA can only capture linear combinations of the raw features. I am pooling across all clients, consistent with the ML-04 pooling decision: client_hash_id is used only for context and grouping, never as a feature, because the goal is archetypes that generalize across accounts rather than describe one client's habits.

Because a clustering task has no label to beat, "the baseline" in this notebook means a linear alternative doing the same job: PCA feeding the same K-Means clustering, on the same features, same split, and same k. If the Autoencoder cannot beat plain PCA on reconstruction error and cluster separation, the added complexity of a neural network is not earning its place, and that itself is a valid, reportable finding. This resolves the open question flagged in my working log about what "baseline" means for an unsupervised capstone.

This notebook uses an impression floor of ≥10 (versus ≥50 used in the ML-07 baseline), since a clustering task benefits from including lower-traffic content as legitimate archetype candidates rather than filtering them out — a floor calibrated for a ranking/scoring task doesn't transfer directly to an unsupervised discovery task. This floor keeps 120,475 of 319,759 addressable content items

In [ ]:
# CODE CELL 1
# Purpose: Resolve repository root, setup gated DuckDB HF connection, run floor & join diagnostics,
# build or load the aggregated first-half month=2026-03 feature frame (with LEFT JOIN and 
# COALESCE for dim_content nulls), separate context columns, and describe stats.

import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import duckdb

# Fix random state universally
RANDOM_STATE = 42

# 1. Safely resolve repo root (walking up from notebook cwd until .env is found)
current_dir = Path.cwd().resolve()
repo_root = current_dir
while not (repo_root / '.env').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

if not (repo_root / '.env').exists():
    raise FileNotFoundError("Could not find .env file; ensure you are running within the repo tree.")

load_dotenv(repo_root / '.env')
hf_token = os.environ.get("HF_TOKEN")

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
# Cache path for the feature frame
cache_path = output_dir / "w05_features_2026_03_half_floor10.parquet" 

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

# --- DIAGNOSTIC BLOCK ---
print("--- DIAGNOSTIC: IMPRESSION FLOOR IMPACT ---")

# Unfloored fact table count
query_unfloored = """
SELECT COUNT(*) FROM (
    SELECT content_hash_id 
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
)
"""
count_unfloored = con.execute(query_unfloored).fetchone()[0]

# Fact table only count (floored >= 10 impressions)
query_fact = """
SELECT COUNT(*) FROM (
    SELECT content_hash_id 
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
)
"""
count_fact = con.execute(query_fact).fetchone()[0]

pct_kept = (count_fact / count_unfloored) * 100 if count_unfloored > 0 else 0

print(f"Unfloored fact distinct content_hash_id: {count_unfloored}")
print(f"Floored (>=10 imp) distinct content_hash_id: {count_fact}")
print(f"Percentage kept by impression floor: {pct_kept:.2f}%")
print("--------------------------------------------------\n")

print("--- DIAGNOSTIC: INNER JOIN vs FACT-ONLY COUNTS ---")

# Inner join count
query_inner = """
SELECT COUNT(*) FROM (
    SELECT f.content_hash_id 
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
        ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date <= DATE '2026-03-15'
    GROUP BY f.content_hash_id
    HAVING SUM(f.gsc_impressions) >= 10
)
"""
count_inner = con.execute(query_inner).fetchone()[0]

diff = count_fact - count_inner
pct_missing = (diff / count_fact) * 100 if count_fact > 0 else 0

print(f"Fact-only (floored) distinct content_hash_id: {count_fact}")
print(f"INNER JOIN distinct content_hash_id:  {count_inner}")
print(f"Difference: {diff} items dropped by INNER JOIN ({pct_missing:.2f}% of fact items have no dim_content match)")
print("--------------------------------------------------\n")
# ------------------------

# 2. Build or Load Data
if cache_path.exists():
    print(f"Loading cached feature frame from: {cache_path}")
    df = con.execute(f"SELECT * FROM read_parquet('{cache_path}')").df()
else:
    print("Cached feature frame not found. Building via DuckDB...")

    # DuckDB Query: filter to month=2026-03 days 1-15, compute GSC metrics globally,
    # conditionally compute GA4 metrics, use LEFT JOIN to dim_content, and COALESCE metadata nulls.
    query = """
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id) AS client_hash_id,
        SUM(f.gsc_impressions) AS impressions_first_half,
        SUM(f.gsc_clicks) AS clicks_first_half,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr,
        AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position ELSE NULL END) AS avg_position,
        
        COALESCE(
            SUM(CASE WHEN f.ga4_data_available THEN f.ga4_engaged_sessions ELSE NULL END) * 1.0 / 
            NULLIF(SUM(CASE WHEN f.ga4_data_available THEN f.ga4_sessions ELSE NULL END), 0), 
            0.0
        ) AS engagement_rate,
        
        AVG(CASE WHEN f.ga4_data_available THEN 1.0 ELSE 0.0 END) AS has_ga4_coverage,
        
        COALESCE(ANY_VALUE(c.search_volume), 0) AS search_volume,
        COALESCE(ANY_VALUE(c.competition), 0) AS competition,
        COALESCE(ANY_VALUE(c.backlinks), 0) AS backlinks,
        COALESCE(ANY_VALUE(c.word_count), 0) AS word_count,
        COALESCE(ANY_VALUE(c.char_count), 0) AS char_count,
        
        ANY_VALUE(CASE WHEN c.backlinks IS NOT NULL THEN 1.0 ELSE 0.0 END) AS has_backlink_data,
        ANY_VALUE(CASE WHEN c.word_count IS NOT NULL THEN 1.0 ELSE 0.0 END) AS has_word_count_data,
        
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-15') AS content_age_days
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    LEFT JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
        ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date <= DATE '2026-03-15'
    GROUP BY f.content_hash_id
    HAVING SUM(f.gsc_impressions) >= 10
    """
    
    df = con.execute(query).df()
    
    # Drop rows that legally should not be null after COALESCE protections.
    # This safely catches `avg_position` nulls (gsc traffic but position <=0 entirely) 
    # and any items completely missing dim_content (causing `content_age_days` to be NaN).
    df = df.dropna().reset_index(drop=True)
    con.register('df_to_cache', df)
    con.execute(f"COPY df_to_cache TO '{cache_path}' (FORMAT PARQUET)")
    print(f"Saved feature frame to absolute path: {cache_path.resolve()}")

# 3. Define and cast features to prevent pandas mixed-type evaluation issues
context_cols = ['content_hash_id', 'client_hash_id']
feature_cols = [c for c in df.columns if c not in context_cols]
df[feature_cols] = df[feature_cols].astype(float)

# Benchmark validation
row_count = len(df)
print(f"\nFinal row count: {row_count}")

# We adjust benchmark language to reflect the new knowledge about the ~92k count
benchmark_min, benchmark_max = 150675, 151981
if benchmark_min <= row_count <= benchmark_max:
    print(f"SUCCESS: Row count is within the originally expected benchmark range ({benchmark_min} - {benchmark_max}).")
else:
    print(f"INFO: Row count ({row_count}) differs from the {benchmark_min}-{benchmark_max} baseline. This confirms the aggressive filtering of the >=10 impressions floor and LEFT JOIN correction.")

print("\n--- Missing Data Flag Value Counts ---")
print("has_backlink_data (1.0 = Present, 0.0 = Missing & Filled 0):")
print(df['has_backlink_data'].value_counts(dropna=False))

print("\nhas_word_count_data (1.0 = Present, 0.0 = Missing & Filled 0):")
print(df['has_word_count_data'].value_counts(dropna=False))

print(f"\nFeature list: {feature_cols}")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am using a grouped train/test split by client_hash_id, not a random row split and not a time split. A random split would let content items from the same client leak into both train and test, and since pages from one client can share hidden characteristics (niche, writing style, publishing cadence), the model could quietly memorize client-level patterns instead of learning genuine archetypes — the same grouped-split logic used for the supervised baseline in ML-07 applies here. I am not using a time split, because this notebook does not predict a future outcome; it groups pages by their current signal profile within month=2026-03. The honest question a client-grouped split answers is: do the archetypes discovered on one set of clients still hold — same latent structure, similar cluster shapes — when applied to a completely unseen client's pages? That directly tests the pooling rationale from ML-04 instead of assuming it.

The test set reflects 9 held-out clients — client-grouped by design, but a small enough test population that a different random seed could produce a meaningfully different result. This is disclosed rather than hidden behind a single silhouette number.

In [ ]:
# CODE CELL 2
# Purpose: Execute a group-aware split (80/20) based on client_hash_id to prevent data leakage 
# across train/test sets, ensuring models generalize to unseen clients.

from sklearn.model_selection import GroupShuffleSplit

# Perform group split keeping clients mutually exclusive
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

# Post-split assertion
train_clients = set(train_df['client_hash_id'])
test_clients = set(test_df['client_hash_id'])
intersection = train_clients.intersection(test_clients)

# Assert absolutely no client overlap
assert len(intersection) == 0, f"Critical Split Leakage! {len(intersection)} clients in both sets."
print("Assertion Passed: Intersection of client_hash_id between train and test sets is strictly empty.")

print(f"Train Set: {len(train_df)} rows, {len(train_clients)} unique clients")
print(f"Test Set:  {len(test_df)} rows, {len(test_clients)} unique clients")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The comparison table shows a genuine disagreement between the two evaluation metrics: the Autoencoder achieves dramatically lower reconstruction error than PCA (0.0048 vs. 0.155 MSE on train), but PCA produces better-separated clusters (silhouette 0.372 vs. 0.271 on train, 0.275 vs. 0.243 on test). Two supplementary diagnostics were run to understand this disagreement rather than simply report it.

First, an ablation retrained the Autoencoder with a visibly tighter bottleneck (3 units instead of 8, against the same 14-feature input). Forcing tighter compression caused reconstruction MSE to rise sharply (0.0048 → 0.270) while silhouette nearly doubled (0.271 → 0.517, sampled). This confirms that the original 8-unit bottleneck had more capacity than the underlying structure required, and used that slack to minimize reconstruction loss rather than being forced to discover a compact, cluster-organized representation.

Second, a linear-reconstructability check tested whether the Autoencoder's bottleneck was behaving close to a linear (near-identity) encoding, as initially suspected. It was not: a linear map recovers only 76.3% of the raw feature variance from the AE's bottleneck (R²=0.7628 train), less than the 84.5% recoverable from PCA's own linear components (R²=0.8448 train, as expected since PCA is linear by construction). This rules out the simplest version of the near-identity explanation — the Autoencoder is doing something genuinely non-linear, not merely passing information through — but that non-linearity is not the kind that produces well-separated clusters for this feature set at this bottleneck width.

Together, these results point to a capacity/objective mismatch rather than a flaw in the non-linear approach itself: given a bottleneck wide enough to minimize reconstruction loss, the network optimizes for reconstruction, not for cluster separability, since only the reconstruction objective is in its loss function. Notably, the tighter 3-unit variant's silhouette (0.517, sampled) exceeds PCA's — suggesting a more aggressively compressed Autoencoder may be a stronger direction than either the original 8-unit model or plain PCA, at the cost of reconstruction fidelity. This is reported as a directional observation, not a re-run comparison, since it used the faster sampled silhouette scoring rather than the exact metric used in the main table. The primary result stands as reported: for this feature set and this bottleneck width, PCA's linear projection organizes the pooled SEO signals into more separable clusters than the Autoencoder does, even though the Autoencoder reconstructs the input far more faithfully.

In [ ]:
# CODE CELL 3
# Purpose: Standardize features, fit PCA (baseline) vs. Autoencoder (MLPRegressor), manually extract 
# latent bottleneck representations, scan for optimal KMeans 'k', and print final comparison deliverable.
# Includes bottleneck ablation and linear reconstructability diagnostics.

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, silhouette_score
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings('ignore', category=FutureWarning) # Suppress sklearn KMeans n_init warnings

X_train_raw = train_df[feature_cols]
X_test_raw = test_df[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
print("✓ Scaler fit and transform complete.")

# --- BASELINE: PCA ---
print("\nPCA Base: Opting for k=8 based on instructions to capture sufficient variance across these heterogeneous features.")
pca = PCA(n_components=8, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print("✓ PCA fit and transform complete.")

pca_train_recon = pca.inverse_transform(X_train_pca)
pca_test_recon = pca.inverse_transform(X_test_pca)
pca_train_mse = mean_squared_error(X_train_scaled, pca_train_recon)
pca_test_mse = mean_squared_error(X_test_scaled, pca_test_recon)

# Scan for best PCA k (5 to 8) using silhouette
print("\nScanning for optimal PCA KMeans k (5 to 8)...")
best_pca_k, best_pca_sil = 5, -1
pca_kmeans_models = {}

for k in range(5, 9):
    with joblib.parallel_backend('threading'):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
        preds = km.fit_predict(X_train_pca)
        pca_kmeans_models[k] = km
    
    # Efficiency Fix 2: Approximate silhouette using sample_size during scan
    sil = silhouette_score(X_train_pca, preds, sample_size=10000, random_state=RANDOM_STATE)
    print(f"  - PCA k={k} silhouette (sampled): {sil:.4f}")
    if sil > best_pca_sil:
        best_pca_k, best_pca_sil = k, sil

print(f"✓ Winning PCA k: {best_pca_k} (Sampled Silhouette: {best_pca_sil:.4f})")

# Efficiency Fix 1: Retrieve fitted model instead of refitting
km_pca = pca_kmeans_models[best_pca_k]

# Exact silhouette for final reporting
pca_train_sil = silhouette_score(X_train_pca, km_pca.labels_)
pca_test_sil = silhouette_score(X_test_pca, km_pca.predict(X_test_pca))

# --- MODEL: Autoencoder via MLPRegressor ---
mlp = MLPRegressor(
    hidden_layer_sizes=(16, 8, 16),
    activation='relu',
    random_state=RANDOM_STATE,
    early_stopping=True
)

print("\nStarting MLPRegressor training...")
mlp.fit(X_train_scaled, X_train_scaled)
print(f"✓ MLPRegressor training complete (iterations: {mlp.n_iter_}).")

ae_train_recon = mlp.predict(X_train_scaled)
ae_test_recon = mlp.predict(X_test_scaled)
ae_train_mse = mean_squared_error(X_train_scaled, ae_train_recon)
ae_test_mse = mean_squared_error(X_test_scaled, ae_test_recon)

# Extract Latent Representation manually using NumPy (simulating ReLU forward pass up to bottleneck)
def extract_bottleneck(X_in, mlp_model):
    # Input -> Hidden Layer 1 
    h1 = np.maximum(0, np.dot(X_in, mlp_model.coefs_[0]) + mlp_model.intercepts_[0])
    # Hidden Layer 1 -> Bottleneck Layer 2 
    bottleneck = np.maximum(0, np.dot(h1, mlp_model.coefs_[1]) + mlp_model.intercepts_[1])
    return bottleneck

X_train_ae = extract_bottleneck(X_train_scaled, mlp)
X_test_ae = extract_bottleneck(X_test_scaled, mlp)
print("✓ Bottleneck representations successfully extracted via manual forward pass.")

# Scan for best AE KMeans k (5 to 8) using silhouette
print("\nScanning for optimal AE KMeans k (5 to 8)...")
best_ae_k, best_ae_sil = 5, -1
ae_kmeans_models = {}

for k in range(5, 9):
    with joblib.parallel_backend('threading'):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
        preds = km.fit_predict(X_train_ae)
        ae_kmeans_models[k] = km
        
    if len(np.unique(preds)) > 1: # Guard against representation collapse
        # Efficiency Fix 2: Approximate silhouette using sample_size during scan
        sil = silhouette_score(X_train_ae, preds, sample_size=10000, random_state=RANDOM_STATE)
        print(f"  - AE k={k} silhouette (sampled): {sil:.4f}")
        if sil > best_ae_sil:
            best_ae_k, best_ae_sil = k, sil
    else:
        print(f"  - AE k={k} failed (representation collapse)")

print(f"✓ Winning AE k: {best_ae_k} (Sampled Silhouette: {best_ae_sil:.4f})")

# Efficiency Fix 1: Retrieve fitted model instead of refitting
km_ae = ae_kmeans_models[best_ae_k]

# Exact silhouette for final reporting
ae_train_sil = silhouette_score(X_train_ae, km_ae.labels_)
ae_test_sil = silhouette_score(X_test_ae, km_ae.predict(X_test_ae))

# Final Deliverable Table (identical train/test scope confirmed)
deliverable_df = pd.DataFrame({
    'Model': ['PCA baseline', 'Autoencoder model'],
    'train_recon_mse': [pca_train_mse, ae_train_mse],
    'test_recon_mse': [pca_test_mse, ae_test_mse],
    'train_silhouette': [pca_train_sil, ae_train_sil],
    'test_silhouette': [pca_test_sil, ae_test_sil],
    'k': [best_pca_k, best_ae_k]
}).set_index('Model')

print("\n--- Model Evaluation Comparison ---")
print(deliverable_df)


# ==============================================================================
# NEW DIAGNOSTIC BLOCKS
# ==============================================================================

print("\n--- Supplementary Diagnostics: Testing the Near-Identity Hypothesis ---")

# --- DIAGNOSTIC 1: Bottleneck Ablation ---
print("\nStarting Tighter MLPRegressor (8, 3, 8) training for ablation...")
mlp_tight = MLPRegressor(
    hidden_layer_sizes=(8, 3, 8),
    activation='relu',
    random_state=RANDOM_STATE,
    early_stopping=True
)
mlp_tight.fit(X_train_scaled, X_train_scaled)
print(f"✓ Tighter MLPRegressor training complete (iterations: {mlp_tight.n_iter_}).")

tight_train_recon = mlp_tight.predict(X_train_scaled)
tight_test_recon = mlp_tight.predict(X_test_scaled)
tight_train_mse = mean_squared_error(X_train_scaled, tight_train_recon)
tight_test_mse = mean_squared_error(X_test_scaled, tight_test_recon)

X_train_tight = extract_bottleneck(X_train_scaled, mlp_tight)
X_test_tight = extract_bottleneck(X_test_scaled, mlp_tight)
print("✓ Tighter bottleneck representations extracted via manual forward pass.")

best_tight_k, best_tight_sil = 5, -1
tight_kmeans_models = {}

for k in range(5, 9):
    with joblib.parallel_backend('threading'):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
        preds = km.fit_predict(X_train_tight)
        tight_kmeans_models[k] = km
        
    if len(np.unique(preds)) > 1:
        sil = silhouette_score(X_train_tight, preds, sample_size=10000, random_state=RANDOM_STATE)
        if sil > best_tight_sil:
            best_tight_k, best_tight_sil = k, sil

km_tight = tight_kmeans_models[best_tight_k]
tight_train_full_sil = silhouette_score(X_train_tight, km_tight.labels_)

print("\n[SUPPLEMENTARY DIAGNOSTIC: Ablation Comparison]")
print(f"Main AE (8-unit)  -> Train MSE: {ae_train_mse:.4f}, Train Silhouette (k={best_ae_k}): {ae_train_sil:.4f}")
print(f"Tight AE (3-unit) -> Train MSE: {tight_train_mse:.4f}, Train Silhouette (k={best_tight_k}): {tight_train_full_sil:.4f}")


# --- DIAGNOSTIC 2: Linear Reconstructability Check ---
print("\nStarting Linear Reconstructability checks...")

# AE Reconstructability
lr_ae_train = LinearRegression().fit(X_train_ae, X_train_scaled)
r2_ae_train = lr_ae_train.score(X_train_ae, X_train_scaled)

lr_ae_test = LinearRegression().fit(X_test_ae, X_test_scaled)
r2_ae_test = lr_ae_test.score(X_test_ae, X_test_scaled)

# PCA Reconstructability (Benchmark)
lr_pca_train = LinearRegression().fit(X_train_pca, X_train_scaled)
r2_pca_train = lr_pca_train.score(X_train_pca, X_train_scaled)

lr_pca_test = LinearRegression().fit(X_test_pca, X_test_scaled)
r2_pca_test = lr_pca_test.score(X_test_pca, X_test_scaled)

print("✓ Linear Reconstructability checks complete.")

print(f"\nLinear reconstructability of AE bottleneck (R²): Train={r2_ae_train:.4f}, Test={r2_ae_test:.4f} — a value near 1.0 suggests the bottleneck can be nearly recovered by a LINEAR map, consistent with a near-identity / near-linear encoding rather than genuine non-linear compression.")
print(f"Benchmark: Linear reconstructability of PCA components (R²): Train={r2_pca_train:.4f}, Test={r2_pca_test:.4f}")

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

There is no "wrong prediction" in clustering the way there is in classification, so I am reading errors as low-confidence assignments: pages sitting near a cluster boundary (a silhouette score close to zero or negative for that row) are the unsupervised equivalent of the model being unsure, worth inspecting by hand the same way ML-07's top-10 review was. I also map each cluster's profile back to the protect/improve/merge/prune vocabulary already established in the ML-07 baseline, as a provisional first pass — the real action-per-archetype logic belongs in the Week-7 action playbook, not here. Two known data-quality issues from earlier notebooks get re-checked against the clusters rather than assumed away: the 4.21% GA4 survival rate (a cluster built heavily on GA4-derived features might just be separating "has GA4 data" from "doesn't"), and the exact-zero-CTR tracking-integrity pattern found in ML-07's top 10 (a cluster dominated by those rows may describe a tracking bug, not a content archetype).

The error analysis confirms two risks flagged in advance. First, clusters 1 (n=69) and 5 (n=14) fall well below a reasonable sample-size floor for a trustworthy archetype and are dominated by the same zero-CTR-despite-traffic tracking-integrity pattern first observed in the ML-07 baseline review (64.3% and 31.9% of rows respectively) alongside unusually high search_volume and almost no backlink/word_count metadata. These two clusters are best read as a data-quality signal — likely batches of pages with broken tracking — rather than genuine content archetypes, and are reported separately from the main archetype narrative rather than folded into it. Consistent with this, these two clusters also account for the large majority of the lowest-silhouette (most boundary/ambiguous) test rows, corroborating the size and disclosure evidence independently.

Second, every cluster shows a substantial fraction of rows with near-zero GA4 engagement (57%–99%), consistent with the project's known 4.21% row-level GA4 survival rate — this is a baseline limitation of the data, not specific to any one cluster. Cluster 7 is a notable exception at 36.3%, alongside the strongest engagement rate and traffic profile of any well-populated cluster. Whether Cluster 7 represents a genuinely more-engaging content archetype or is partly an artifact of better GA4 instrumentation among the clients whose content lands there cannot be fully separated with the current features, and is disclosed as an open limitation rather than resolved.

A provisional protect/improve/merge/prune mapping, to be refined in the action-playbook stage: Cluster 7 as a protect candidate (highest engagement, best tracking coverage, moderate traffic); Clusters 0 and 3 as improve candidates (young content, complete metadata, decent position, but low current traffic); Clusters 2 and 6 as prune candidates (very low traffic, the weakest average positions of any cluster, near-total GA4 absence); Clusters 1 and 5 excluded from the archetype framing entirely and flagged instead as likely tracking-integrity artifacts warranting data-pipeline investigation rather than content action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.